# Atelier AI — Behavioral EDA

## Objective

Investigate customer activity, article popularity, repeat purchases
and temporal variation to formulate testable hypotheses for candidate
generation and feature engineering.

The analysis uses transactions strictly before 2020-09-09.
Validation and local-test targets remain outside the behavioral
analysis scope.

Source characteristics and handling decisions are documented in the
[data card](../docs/data_card.md).
Temporal boundaries follow the
[evaluation strategy](../docs/evaluation.md).

The notebook runs independently from the
[data-foundation notebook](data_foundation.ipynb).

## 1. Environment

The project virtual environment provides the analysis dependencies
and the installed atelier_ai package.

Each analysis must run from a fresh kernel without relying on
variables created in another notebook.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import atelier_ai

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Package location:", Path(atelier_ai.__file__).resolve())
print("pandas version:", pd.__version__)

Python version: 3.12.14
Python executable: /Users/oumar/Developer/portfolio/atelier-ai-project/.venv/bin/python
Package location: /Users/oumar/Developer/portfolio/atelier-ai-project/src/atelier_ai/__init__.py
pandas version: 3.0.6


## 2. Analysis Contract

### 2.1. Time Boundaries and Population

The analysis interval is [2018-09-20, 2020-09-09):
the start is included and the end is excluded.

The behavioral population consists of customers with at least one
observed transaction within this interval. Inclusion does not depend
on purchases in the validation or test periods.

Initial behavioral analyses use all eligible transaction rows.
A future modeling cohort and modeling lookback are separate choices.

First and last purchases refer to the first and last observed records
within the declared interval. They do not establish lifetime behavior,
registration dates or article launch dates.

Customers without an observed history require separate coverage
diagnostics and are not represented by active-customer averages.

Analyses used to design features exclude validation and test targets.
Features calculated later at a historical prediction cutoff must use
only records strictly before that particular cutoff.

In [2]:
PROJECT_ROOT = Path(atelier_ai.__file__).resolve().parents[2]
RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw" / "hm"

EDA_START = pd.Timestamp("2018-09-20")
EDA_END_EXCLUSIVE = pd.Timestamp("2020-09-09")

CHUNK_ROWS = 100_000

if not EDA_START < EDA_END_EXCLUSIVE:
    raise ValueError(
        "EDA start must precede its exclusive end."
    )

if not (RAW_DATA_DIRECTORY / "transactions_train.csv").is_file():
    raise FileNotFoundError(
        "Required source file: transactions_train.csv"
    )

print("Start included:", EDA_START.date())
print("End excluded:", EDA_END_EXCLUSIVE.date())
print(
    "Calendar days:",
    (EDA_END_EXCLUSIVE - EDA_START).days,
)
print("Transaction source found.")

Start included: 2018-09-20
End excluded: 2020-09-09
Calendar days: 720
Transaction source found.


### 2.2. Measurement Definitions

| Measure | Definition | Interpretation boundary |
|---|---|---|
| Transaction rows | Number of retained source rows | Not an identified order or basket count |
| Observed purchase days | Distinct transaction dates for a customer | Several rows on the same day count as one day |
| Distinct purchased articles | Distinct article_id values within the announced group and period | Repeated purchases do not increase this count |
| Observed buyers | Distinct customers purchasing an article within the interval | Buyer counts cannot be summed across articles to obtain distinct customers |
| Customer recency | Calendar-day difference between the reference cutoff and the last observed purchase | A purchase on 2020-09-08 has a recency of one day at the 2020-09-09 cutoff |

Source transaction repetitions are retained.
Distinct article relevance for ranking evaluation is defined separately.

Every proportion must specify its denominator.
Transaction-based shares and customer-based shares answer different
questions.

### 2.3. Aggregate Requirements

These tables describe the analytical requirements.
They will be constructed progressively during the corresponding
analysis stages.

| Aggregate | One row represents | Required measures |
|---|---|---|
| Daily activity | One calendar date | Transaction-row count and source observation flag |
| Customer activity | One observed customer | Rows, purchase days, distinct articles, first and last observed purchase, recency |
| Article activity | One observed article | Rows, distinct buyers, first and last observed purchase |
| Customer–article history | One observed customer–article pair | Rows, distinct purchase days, first and last observed purchase |
| Category activity | One date and classification code | Transaction rows and period totals needed for category shares |
| Customer–category preferences | One observed customer and classification code | Rows, distinct articles and customer totals needed for preference shares |

Distinct counts must remain exact across chunk boundaries.
Large customer–article tables are not assumed to fit simultaneously
in memory.

### 2.4. Metadata and Interpretation Policy

- Preserve exact string identifiers and retained transaction repetitions.
- Use article_id as the recommendation identifier.
- Treat product_code as a grouping field.
- Filter eligible transactions before behavioral aggregation.
- Validate many-to-one metadata joins and reconcile row counts.
- Classification-based descriptions refer to the acquired metadata
  snapshot; historical availability is not established.
- Do not use customer fields with unresolved observation dates to
  define behavioral segments in this initial analysis.
- Preserve unresolved categories and missing-value meanings.
- Retain source price units and sales-channel codes.
- Do not infer stock availability, exposure, causality or business impact.

In [3]:
from contextlib import closing

from atelier_ai.data_io import read_source_chunks
from atelier_ai.temporal import interval_mask

## 3. Eligible Transaction History

### 3.1. Chunked Date Reading

The initial scope check reads only the transaction-date column.

Each source chunk is processed independently:
1. Parse dates into a temporary datetime representation.
2. Select the declared interval [EDA_START, EDA_END_EXCLUSIVE).
3. Yield the corresponding source date rows.

Source files and retained date strings remain unchanged.
Missing or unparseable dates stop the calculation.

Reading is incremental. Filtered chunks are not accumulated into
a complete transaction DataFrame.

The reader may traverse dates outside the analysis interval.
Only eligible rows are passed to downstream aggregation.

In [4]:
def iter_eda_date_chunks():
    """Yield source date rows within the declared EDA interval."""
    with read_source_chunks(
        RAW_DATA_DIRECTORY / "transactions_train.csv",
        columns=["t_dat"],
        chunk_rows=CHUNK_ROWS,
    ) as reader:
        for chunk in reader:
            parsed_dates = pd.to_datetime(
                chunk["t_dat"],
                format="%Y-%m-%d",
                errors="raise",
            )

            if parsed_dates.isna().any():
                raise ValueError(
                    "Transaction dates must be complete."
                )

            eligible = interval_mask(
                parsed_dates,
                EDA_START,
                EDA_END_EXCLUSIVE,
            )

            eligible_dates = chunk.loc[eligible, ["t_dat"]]

            if not eligible_dates.empty:
                yield eligible_dates

### 3.2. Limited Reader Check

Inspect the first nonempty eligible chunk to verify the reader,
date conversion and interval selection.

The summary describes this chunk only.
It does not establish complete-file coverage or total transaction counts.

Close the generator explicitly after this early stop so that its
underlying source reader is also closed.

In [ ]:
with closing(iter_eda_date_chunks()) as eda_chunks:
    first_eda_chunk = next(eda_chunks, None)

if first_eda_chunk is None:
    raise ValueError(
        "No transaction dates were found in the EDA interval."
    )

preview_dates = pd.to_datetime(
    first_eda_chunk["t_dat"],
    format="%Y-%m-%d",
    errors="raise",
)

preview_in_scope = interval_mask(
    preview_dates,
    EDA_START,
    EDA_END_EXCLUSIVE,
)

if not preview_in_scope.all():
    raise ValueError(
        "The filtered chunk contains dates outside the EDA interval."
    )

first_chunk_summary = pd.DataFrame([{
    "retained_rows": len(first_eda_chunk),
    "first_date": preview_dates.min().date(),
    "last_date": preview_dates.max().date(),
    "all_dates_in_scope": bool(preview_in_scope.all()),
}])

first_chunk_summary

,retained_rows,first_date,last_date,all_dates_in_scope
0,100000,2018-09-20,2018-09-22,True
